# Build Custom Layers & Modules Example

Build custom layers and modules with JAX, using Flax.

In Flax, every layer (`nnx.Linear`, `nnx.Conv`, ...) and every model is a
`nnx.Module`: a Python class holding its own weights as `nnx.Param`
attributes. A custom layer is built the same way, so it can be mixed
freely with the built-in layers, and trained like any other model.

This example is using the MNIST database of handwritten digits
(http://yann.lecun.com/exdb/mnist/).

- Author: Aymeric Damien
- Project: [https://github.com/aymericdamien/JAX-Examples/](https://github.com/aymericdamien/JAX-Examples/)

In [1]:
import os
import sys

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

# Make the shared helpers of the 'common' folder importable.
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from common.datasets import load_mnist, batch_iterator

In [2]:
# MNIST dataset parameters.
num_classes = 10  # 0 to 9 digits
num_features = 784  # 28*28

# Training parameters.
learning_rate = 0.01
training_steps = 500
batch_size = 256
display_step = 50

In [3]:
# Prepare MNIST data.
(x_train, y_train), (x_test, y_test) = load_mnist()
# Convert to float32.
x_train, x_test = np.array(x_train, np.float32), np.array(x_test, np.float32)
# Flatten images to 1-D vector of 784 features (28*28).
x_train, x_test = x_train.reshape([-1, num_features]), x_test.reshape([-1, num_features])
# Normalize images value from [0, 255] to [0, 1].
x_train, x_test = x_train / 255., x_test / 255.

In [4]:
# Shuffle and batch data (a simple data loader, see 'common/datasets.py').
train_data = batch_iterator(x_train, y_train, batch_size)

## Create a custom layer

Build a custom layer with inner-variables.

In [5]:
# Create a custom layer, extending Flax 'Module' class.
# Layer compute: y = relu(W * x + b)
class CustomLayer1(nnx.Module):

    # Layer arguments. Flax layers create their weights right away, so the
    # number of input features must be given (there is no lazy 'build').
    def __init__(self, num_inputs, num_units, rngs):
        # Create weight variables for this layer. Wrapping an array in
        # 'nnx.Param' marks it as a trainable parameter of the layer.
        # 'rngs.params()' provides a new random key for the initialization.
        self.weight = nnx.Param(
            0.05 * jax.random.normal(rngs.params(), (num_inputs, num_units)))
        self.bias = nnx.Param(jnp.zeros(num_units))

    # Set forward pass (a 'Param' can be used like a regular array).
    def __call__(self, inputs):
        x = jnp.matmul(inputs, self.weight)
        x = x + self.bias
        return nnx.relu(x)

## Create another custom layer

Build another custom layer with inner Flax 'layers'.

In [6]:
# Create a custom layer, extending Flax 'Module' class.
# Custom layer: 2 Fully-Connected layers with residual connection.
class CustomLayer2(nnx.Module):

    # Layer arguments.
    def __init__(self, num_units, rngs):
        # Note: a custom layer can include any other Flax layers, their
        # weights are then part of the custom layer.
        self.inner_layer1 = nnx.Linear(num_units, 1, rngs=rngs)
        self.inner_layer2 = nnx.Linear(1, num_units, rngs=rngs)

    def __call__(self, inputs):
        x = self.inner_layer1(inputs)
        x = nnx.relu(x)
        x = self.inner_layer2(x)
        return x + inputs

## Build Model

In [7]:
# Create Flax Model.
class CustomNet(nnx.Module):

    def __init__(self, rngs):
        # Use custom layers created above.
        self.layer1 = CustomLayer1(num_features, 64, rngs=rngs)
        self.layer2 = CustomLayer2(64, rngs=rngs)
        # And a built-in layer, for the output.
        self.out = nnx.Linear(64, num_classes, rngs=rngs)

    # Set forward pass.
    def __call__(self, x):
        x = self.layer1(x)
        x = nnx.relu(x)
        x = self.layer2(x)
        # Return the raw scores (logits): the loss function applies softmax.
        return self.out(x)

# Build neural network model.
custom_net = CustomNet(rngs=nnx.Rngs(0))

In [8]:
# Display the shape of every model parameter: the weights of the custom
# layers are tracked like those of the built-in layers.
for path, param in nnx.to_flat_state(nnx.state(custom_net, nnx.Param)):
    print('/'.join(map(str, path)), param.shape)

layer1/bias (64,)
layer1/weight (784, 64)
layer2/inner_layer1/bias (1,)
layer2/inner_layer1/kernel (64, 1)
layer2/inner_layer2/bias (64,)
layer2/inner_layer2/kernel (1, 64)
out/bias (10,)
out/kernel (64, 10)


In [9]:
# Cross-Entropy loss function.
# Note that this will apply 'softmax' to the logits.
def cross_entropy(y_pred, y_true):
    loss = optax.softmax_cross_entropy_with_integer_labels(logits=y_pred, labels=y_true)
    return jnp.mean(loss)

# Accuracy metric.
def accuracy(y_pred, y_true):
    # Predicted class is the index of highest score in prediction vector (i.e. argmax).
    correct_prediction = jnp.equal(jnp.argmax(y_pred, 1), y_true)
    return jnp.mean(correct_prediction.astype(jnp.float32))

# Adam optimizer.
optimizer = nnx.Optimizer(custom_net, optax.adam(learning_rate), wrt=nnx.Param)

In [10]:
# Optimization process.
# @nnx.jit compiles the function with XLA, so it runs much faster
# (see '1_Introduction/transformations.py').
@nnx.jit
def run_optimization(custom_net, optimizer, x, y):
    # Define the loss as a function of the model, to differentiate it.
    def loss_fn(custom_net):
        pred = custom_net(x)
        return cross_entropy(pred, y)

    # Compute gradients (for the custom layers weights too).
    gradients = nnx.grad(loss_fn)(custom_net)

    # Update the model parameters following gradients.
    optimizer.update(custom_net, gradients)

In [11]:
# Run training for the given number of steps.
for step in range(1, training_steps + 1):
    # Get the next batch of data.
    batch_x, batch_y = next(train_data)
    # Run the optimization to update the model parameters.
    run_optimization(custom_net, optimizer, batch_x, batch_y)

    if step % display_step == 0:
        pred = custom_net(batch_x)
        loss = cross_entropy(pred, batch_y)
        acc = accuracy(pred, batch_y)
        print("step: %i, loss: %f, accuracy: %f" % (step, loss, acc))

step: 50, loss: 0.259348, accuracy: 0.914062
step: 100, loss: 0.300316, accuracy: 0.917969
step: 150, loss: 0.203259, accuracy: 0.957031
step: 200, loss: 0.113037, accuracy: 0.972656
step: 250, loss: 0.078483, accuracy: 0.984375
step: 300, loss: 0.085576, accuracy: 0.980469


step: 350, loss: 0.096167, accuracy: 0.976562
step: 400, loss: 0.174345, accuracy: 0.953125
step: 450, loss: 0.143938, accuracy: 0.964844
step: 500, loss: 0.077602, accuracy: 0.988281


In [12]:
# Test model on validation set.
pred = custom_net(x_test)
print("Test Accuracy: %f" % accuracy(pred, y_test))

Test Accuracy: 0.965800
